# Requirement 3 — Approach 1: DL-Based Feature Extraction + SVM Classifier
**Course:** CAI3105/CS460 — Deep Learning  
**Dataset:** Brain Tumor MRI (Kaggle)  
**Pre-trained Model:** EfficientNet-B0  
**ML Classifier:** SVM (Linear Kernel)

## Step 1 — Mount Google Drive & Extract Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

zip_path = '/content/drive/MyDrive/Brain_Tumor/archive.zip'
extract_path = '/content/brain_tumor_data'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted!")
print("Training classes:", os.listdir(extract_path + '/Training'))
print("Testing classes :", os.listdir(extract_path + '/Testing'))


## Step 2 — Import Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

print("TensorFlow version:", tf.__version__)
print("All libraries imported successfully.")


## Step 3 — Data Preprocessing & Augmentation

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

TRAIN_DIR = '/content/brain_tumor_data/Training'
TEST_DIR  = '/content/brain_tumor_data/Testing'

# Training: with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    fill_mode='nearest'
)

# Testing: only rescale (no augmentation)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False   # shuffle=False for consistent feature extraction
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

CLASS_NAMES = list(train_generator.class_indices.keys())
print("Classes:", CLASS_NAMES)
print("Training samples:", train_generator.samples)
print("Testing samples :", test_generator.samples)


## Step 4 — Build EfficientNet-B0 Feature Extractor (Frozen)

In [ ]:
# Load EfficientNet-B0 without the top classification head
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze all layers — we use it ONLY as a feature extractor
for layer in base_model.layers:
    layer.trainable = False

# Add Global Average Pooling to get a 1D feature vector per image
x = base_model.output
x = GlobalAveragePooling2D()(x)

feature_extractor = Model(inputs=base_model.input, outputs=x)

print("Feature extractor output shape:", feature_extractor.output_shape)
print("Total parameters:", base_model.count_params())


## Step 5 — Extract Features from Training & Testing Sets

In [ ]:
print("Extracting features from TRAINING set...")
start = time.time()
X_train = feature_extractor.predict(train_generator, verbose=1)
y_train = train_generator.classes
train_time = time.time() - start
print(f"Training features shape : {X_train.shape}")
print(f"Extraction time         : {train_time:.2f} seconds")

print("\nExtracting features from TESTING set...")
start = time.time()
X_test = feature_extractor.predict(test_generator, verbose=1)
y_test = test_generator.classes
test_time = time.time() - start
print(f"Testing features shape  : {X_test.shape}")
print(f"Extraction time         : {test_time:.2f} seconds")


## Step 6 — Train SVM Classifier on Extracted Features

In [ ]:
print("Training SVM classifier (Linear Kernel)...")
start_svm = time.time()

svm_clf = SVC(kernel='linear', C=1.0, random_state=42)
svm_clf.fit(X_train, y_train)

svm_train_time = time.time() - start_svm
print(f"SVM training time: {svm_train_time:.2f} seconds")


## Step 7 — Evaluate SVM Performance

In [ ]:
y_pred = svm_clf.predict(X_test)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted')
rec  = recall_score(y_test, y_pred, average='weighted')
f1   = f1_score(y_test, y_pred, average='weighted')

print("=" * 50)
print("  Approach-1: EfficientNet-B0 + SVM Results")
print("=" * 50)
print(f"  Accuracy  : {acc*100:.2f}%")
print(f"  Precision : {prec*100:.2f}%")
print(f"  Recall    : {rec*100:.2f}%")
print(f"  F1-Score  : {f1*100:.2f}%")
print("=" * 50)

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))


## Step 8 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)
plt.title('Confusion Matrix — Approach 1: EfficientNet-B0 + SVM', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.savefig('/content/confusion_matrix_approach1.png', dpi=150)
plt.show()
print("Confusion matrix saved.")


## Step 9 — Save Results for Comparative Analysis

In [ ]:
import json

results_approach1 = {
    "approach": "EfficientNet-B0 + SVM (Linear Kernel)",
    "accuracy":  round(acc * 100, 2),
    "precision": round(prec * 100, 2),
    "recall":    round(rec * 100, 2),
    "f1_score":  round(f1 * 100, 2),
    "svm_train_time_sec": round(svm_train_time, 2),
    "feature_extraction_time_sec": round(train_time + test_time, 2)
}

with open('/content/drive/MyDrive/Brain_Tumor/results_approach1.json', 'w') as f:
    json.dump(results_approach1, f, indent=4)

print("Results saved:", results_approach1)
